In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from fancyimpute import IterativeImputer, KNN, SoftImpute
from statsmodels.tsa.statespace.kalman_filter import KalmanFilter
from pykalman import KalmanFilter as PyKalmanFilter

# Define the input and output directories
input_directory = '/home/ahmedmas/Projects/Data_imputation/Processed_Data'
output_directory = '/home/ahmedmas/Projects/Data_imputation/model_result'

# Create the output directories if they do not exist
output_imputation_directory = os.path.join(output_directory, 'imputation_results')
os.makedirs(output_imputation_directory, exist_ok=True)

# Function to extract study site from filename
def extract_study_site(filename):
    parts = filename.split('_')
    study_site = parts[1]  # Assuming the study site is the second word in the filename
    return study_site

# Function to calculate missing percentage
def calculate_missing_percentage(df, column):
    total_values = len(df)
    missing_values = df[column].isnull().sum()
    missing_percentage = (missing_values / total_values) * 100
    return missing_percentage, missing_values, total_values

# Function to add hour, day, and month columns
def add_time_columns(df):
    df['Hour'] = df['datetime'].dt.hour
    df['Day'] = df['datetime'].dt.day
    df['Month'] = df['datetime'].dt.month
    return df

# Function to apply various imputation methods
def impute_data(df, method='mean'):
    if method == 'mean':
        imputer = SimpleImputer(strategy='mean')
    elif method == 'median':
        imputer = SimpleImputer(strategy='median')
    elif method == 'most_frequent':
        imputer = SimpleImputer(strategy='most_frequent')
    elif method == 'interpolation':
        return df.interpolate(method='time')
    elif method == 'locf':
        return df.ffill()
    elif method == 'hot_deck':
        return df.apply(lambda x: x.fillna(method='ffill'), axis=0)
    elif method == 'cold_deck':
        return df.apply(lambda x: x.fillna(x.mean()), axis=0)
    elif method == 'mice':
        imputer = IterativeImputer()
    elif method == 'em':
        imputer = SoftImpute()
    elif method == 'spline':
        return df.interpolate(method='spline', order=2)
    elif method == 'kalman':
        kf = PyKalmanFilter(initial_state_mean=0, n_dim_obs=1)
        imputed_data, _ = kf.em(df.values, n_iter=5).smooth(df.values)
        imputed_df = pd.DataFrame(imputed_data, columns=df.columns)
        return imputed_df
    else:
        raise ValueError(f"Unknown imputation method: {method}")
    
    imputed_df = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)
    return imputed_df

# Traverse the directory and process each CSV file
methods = ['mean', 'median', 'most_frequent', 'interpolation', 'locf', 'hot_deck', 'cold_deck', 'mice', 'em', 'spline', 'kalman']
for filename in os.listdir(input_directory):
    if filename.endswith('.csv'):
        study_site = extract_study_site(filename)
        filepath = os.path.join(input_directory, filename)

        # Load the CSV file into a DataFrame
        df = pd.read_csv(filepath)

        # Convert datetime column to datetime type
        df['datetime'] = pd.to_datetime(df['datetime'])

        # Calculate initial missing percentage of PM2.5
        initial_missing_percentage, initial_missing_count, total_values = calculate_missing_percentage(df, 'PM2.5')

        # Create a column to mark originally missing data
        df['missing_info'] = np.where(df['PM2.5'].isnull(), 'OM', '')

        # Determine the number of additional values to remove to achieve 10% missing
        target_missing_percentage = 10
        target_missing_count = int(target_missing_percentage * total_values / 100)
        additional_missing_count = target_missing_count - initial_missing_count

        # Create a column to store removed values
        df['removed_PM2.5'] = np.nan

        if additional_missing_count > 0:
            # Randomly select indices to set as NaN
            non_missing_indices = df[df['PM2.5'].notnull()].index
            if len(non_missing_indices) < additional_missing_count:
                print(f"Not enough non-missing values in {filename} to achieve 10% missing")
                continue
            additional_missing_indices = np.random.choice(non_missing_indices, additional_missing_count, replace=False)
            df.loc[additional_missing_indices, 'removed_PM2.5'] = df.loc[additional_missing_indices, 'PM2.5']
            df.loc[additional_missing_indices, 'PM2.5'] = np.nan

            # Mark the artificially missing data
            df.loc[additional_missing_indices, 'missing_info'] = 'AM'

        # Add columns for hour, day, and month
        df = add_time_columns(df)

        # Drop the specified columns before imputation
        columns_to_drop = ['datetime', 'missing_info', 'removed_PM2.5']
        df_dropped = df.drop(columns=columns_to_drop)

        # Apply various imputation methods and save results
        for method in methods:
            try:
                imputed_df = impute_data(df_dropped, method=method)
            except Exception as e:
                print(f"Error in method {method} for file {filename}: {e}")
                continue

            # Include both observed and predicted PM2.5
            df['Predicted_PM2.5'] = imputed_df['PM2.5']

            # Save the imputed DataFrame to the output directory
            output_filepath = os.path.join(output_imputation_directory, f'imputed_{method}_{filename}')
            df.to_csv(output_filepath, index=False)
            print(f"Processed and saved file: {output_filepath} with method: {method}")

print("Processing complete.")
